# Image classification using CNN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
BASE_DIR  = '/kaggle/input/datasets/puneet6060/intel-image-classification'
TRAIN_DIR = os.path.join(BASE_DIR, 'seg_train', 'seg_train')
TEST_DIR  = os.path.join(BASE_DIR, 'seg_test',  'seg_test')
PRED_DIR  = os.path.join(BASE_DIR, 'seg_pred',  'seg_pred')

CLASSES = sorted(os.listdir(TRAIN_DIR))
print(CLASSES)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, cls in zip(axes.flat, CLASSES):
    img_path = os.path.join(TRAIN_DIR, cls, os.listdir(os.path.join(TRAIN_DIR, cls))[0])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(cls)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Load and resize images

In [ ]:
IMG_SIZE = 64

def load_data(directory):
    images, labels = [], []
    for cls in CLASSES:
        folder = os.path.join(directory, cls)
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname))
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(cls)
    return np.array(images), np.array(labels)

X_train_raw, y_train_raw = load_data(TRAIN_DIR)
X_test_raw,  y_test_raw  = load_data(TEST_DIR)

print(X_train_raw.shape, X_test_raw.shape)

## Normalize and encode labels

In [ ]:
X_train = X_train_raw / 255.0
X_test  = X_test_raw  / 255.0

le = LabelEncoder()
y_train = to_categorical(le.fit_transform(y_train_raw))
y_test  = to_categorical(le.transform(y_test_raw))

print("Classes:", le.classes_)
print("X_train:", X_train.shape, "| y_train:", y_train.shape)

## Visualize what a single image looks like as a pixel array

In [ ]:
sample = X_train[0]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(sample)
axes[0].set_title("Original")

for i, (ch, name) in enumerate(zip([0,1,2], ['Red','Green','Blue'])):
    axes[i+1].imshow(sample[:,:,ch], cmap='gray')
    axes[i+1].set_title(f"{name} channel")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Visualize what convolution does (manual kernel)

In [ ]:
from scipy.ndimage import convolve

gray = np.mean(X_train[0], axis=2)

edge_kernel = np.array([[-1,-1,-1],
                         [-1, 8,-1],
                         [-1,-1,-1]])

convolved = convolve(gray, edge_kernel)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title("Grayscale input")
axes[1].imshow(convolved, cmap='gray')
axes[1].set_title("After edge-detect kernel")
for ax in axes:
    ax.axis('off')
plt.show()

## Build the CNN

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(len(CLASSES), activation='softmax')
])

model.summary()

## Visualize the architecture as a shape flow

In [ ]:
for layer in model.layers:
    if hasattr(layer, 'output'):
        print(f"{layer.name:30s}  output shape: {str(layer.output.shape)}")

## Data augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

sample_batch = X_train[:9]
aug_iter = datagen.flow(sample_batch, batch_size=9, shuffle=False)
aug_batch = next(aug_iter)

fig, axes = plt.subplots(2, 9, figsize=(18, 4))
for i in range(9):
    axes[0, i].imshow(sample_batch[i])
    axes[0, i].axis('off')
    axes[1, i].imshow(np.clip(aug_batch[i], 0, 1))
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Augmented", fontsize=10)
plt.tight_layout()
plt.show()

## Compile and train

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    epochs=20,
    validation_data=(X_test, y_test)
)

## Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## Evaluate and confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test_raw, le.inverse_transform(np.argmax(y_pred, axis=1)))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## Visualize what the model actually learned (feature maps)

In [ ]:
from tensorflow.keras.models import Model

feature_model = Model(inputs=model.inputs, outputs=model.layers[0].output)
fmaps = feature_model.predict(X_test[0:1])

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < fmaps.shape[-1]:
        ax.imshow(fmaps[0, :, :, i], cmap='viridis')
    ax.axis('off')
plt.suptitle("Feature maps after first Conv layer")
plt.tight_layout()
plt.show()

## Predict on a single image (the full pipeline)

In [ ]:
n = 10
indices = np.random.randint(len(X_test), size=n)

fig, axes = plt.subplots(n, 2, figsize=(10, n * 3))

for row, idx in enumerate(indices):
    img = X_test[idx]
    probs = model.predict(img[np.newaxis, ...], verbose=0)[0]

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"True: {y_test_raw[idx]}")
    axes[row, 0].axis('off')

    bars = axes[row, 1].barh(le.classes_, probs, color='steelblue')
    bars[np.argmax(probs)].set_color('tomato')
    axes[row, 1].set_xlim(0, 1)
    axes[row, 1].set_title(f"Predicted: {le.classes_[np.argmax(probs)]}")

plt.tight_layout()
plt.show()